In [2]:
import json
import os
import pandas as pd
import numpy as np
from datetime import datetime
import pandas as pd
import numpy as np

### Function 1: extract sample run info from th-exec logs.
Run right after kicking off the jobs

In [40]:
def extract_run_info_from_bips_log(orderhub_id_list, log_file_dir, sample_info_output_csv_dir):
    run_info_df = pd.DataFrame()
    for orderhub_id in orderhub_id_list:
        print('extracting run info from', orderhub_id)
        try:
            with open(log_file_dir+orderhub_id, 'r') as f:
                log = json.load(f)
                #extract assets 
                log_assets=log['analysis'] 
                                
                #finalize log file
                log=pd.json_normalize(log)
                #select columns from log file
                log=log[['responseBody.jobId','analysis.id','analysis.orderhubId', 'analysis.assay',
                         'analysis.cancerCohort']]
                log=log.rename(columns={"analysis.id": "analysis_id", 
                                        "analysis.orderhubId":"orderhub_id",
                                        "analysis.assay":"assay",
                                        "analysis.cancerCohort":"cancer_cohort",
                                        "responseBody.jobId":"Execution_ID"})
                
            run_info_df=run_info_df.append(log, ignore_index=True)
        except FileNotFoundError:
            print(orderhub_id+'file does not exist')
            pass
    #return run_info_df
    run_info_df.to_csv(sample_info_output_csv_dir, index=False)

In [41]:
#test run example.
os.chdir("/Users/christine.chin/Tickets/Brad/LOD/Altered_Splicing/")
orderhub_ids=pd.read_csv("./jsonList.csv")['ids'].tolist()
extract_run_info_from_bips_log(orderhub_id_list=orderhub_ids,log_file_dir='/Users/christine.chin/Tickets/Brad/LOD/Altered_Splicing/out.json/', 
    sample_info_output_csv_dir='./run_info_LOD_AS_xR_IVD_beta_th_adapter.csv')

extracting run info from out.2024-11-07T19-08-59-118085Z.0.json
extracting run info from out.2024-11-07T19-09-08-063472Z.0.json
extracting run info from out.2024-11-07T19-09-14-138626Z.0.json
extracting run info from out.2024-11-07T19-09-21-296504Z.0.json
extracting run info from out.2024-11-07T19-09-27-833406Z.0.json
extracting run info from out.2024-11-07T19-09-33-874789Z.0.json
extracting run info from out.2024-11-07T19-09-40-085509Z.0.json
extracting run info from out.2024-11-07T19-09-45-921958Z.0.json
extracting run info from out.2024-11-07T19-09-51-519046Z.0.json
extracting run info from out.2024-11-07T19-09-56-904116Z.0.json
extracting run info from out.2024-11-07T19-10-02-389138Z.0.json
extracting run info from out.2024-11-07T19-10-07-978212Z.0.json
extracting run info from out.2024-11-07T19-10-13-818688Z.0.json
extracting run info from out.2024-11-07T19-10-19-442152Z.0.json
extracting run info from out.2024-11-07T19-10-24-942771Z.0.json
extracting run info from out.2024-11-07T

/var/folders/nb/n1jblmfn45bcvgnv2kmpw5c80000gp/T/ipykernel_34439/875164536.py:22: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  run_info_df=run_info_df.append(log, ignore_index=True)
/var/folders/nb/n1jblmfn45bcvgnv2kmpw5c80000gp/T/ipykernel_34439/875164536.py:22: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  run_info_df=run_info_df.append(log, ignore_index=True)
/var/folders/nb/n1jblmfn45bcvgnv2kmpw5c80000gp/T/ipykernel_34439/875164536.py:22: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  run_info_df=run_info_df.append(log, ignore_index=True)
/var/folders/nb/n1jblmfn45bcvgnv2kmpw5c80000gp/T/ipykernel_34439/875164536.py:22: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future versi

###  Function 2: extract state machine info after the run.
Run after jobs are all finished.

To get executions_list.json, run this command to get the execution list from the RNA-onco state machine.

- In bet:

<code>aws stepfunctions list-executions --state-machine-arn arn:aws:states:us-east-1:227117204443:stateMachine:rp-rna-onco-device > executions_list.json </code>

- In val:

`aws stepfunctions list-executions --state-machine-arn arn:aws:states:us-east-1:711036529635:stateMachine:rp-rna-onco-device > executions_list.json
`

Note: run ctv first and export the <code> tempus-bioinformatics-staging </code> role (in bet) or the `tempus-val-bioinformatics` role (in val). Configure the aws regioin to `us-east-1`.


In [2]:
#this function add info from execution list queried from aws to sample sheet.
def udpate_execution_info_after_run(sample_info_before_run_dir, executions_list_json_dir, udpated_sample_info_dir):
    
    with open(executions_list_json_dir, 'r') as f:
        executions_list = pd.json_normalize(json.load(f), 
                                            record_path =['executions']).rename(columns={"name": "executionName"})
        executions_list['duration'] = (pd.to_datetime(executions_list.stopDate) - pd.to_datetime(executions_list.startDate)).dt.total_seconds()/3600
        
    sample_info_before_run=pd.read_csv(sample_info_before_run_dir)
    
    sample_info_after_run=pd.merge(sample_info_before_run, executions_list, how="left", 
                                   on=["executionArn", "stateMachineArn", "executionName"])
    
    sample_info_after_run.to_csv(udpated_sample_info_dir, index=False)

In [8]:
#test run
udpate_execution_info_after_run(sample_info_before_run_dir='./run_info_PCL-00070_b.csv',
                                executions_list_json_dir='./executions_list.json',
                                udpated_sample_info_dir='./run_info_PCL-00070_b.csv')